# P053 — Colab Simulation Runner

Run all cells once.

This notebook now does:
1. setup
2. mount Drive + copy `preprocessed_full.npz`
3. run `fast` → backup
4. run `medium` → backup
5. run `full` → backup
6. print final summary

Drive outputs are saved automatically to:
- `MyDrive/P053_results/fast/`
- `MyDrive/P053_results/medium/`
- `MyDrive/P053_results/full/`

If Colab disconnects, run all cells again. Each mode uses `--checkpoint` and resumes from the last completed day for that mode.

In [1]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: Clone Repo + Install Dependencies
# ═══════════════════════════════════════════════════════════════
import os
import shutil
import subprocess

REPO_URL = "https://github.com/AIML-Engineering-Lab/053_dram_memory_yield_mlops.git"
PROJECT_DIR = "/content/053_memory_yield_predictor"

def build_clone_url(repo_url: str) -> str:
    try:
        from google.colab import userdata
        github_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        github_token = None

    if github_token and repo_url.startswith("https://github.com/"):
        return repo_url.replace("https://", f"https://{github_token}@")
    return repo_url

clone_url = build_clone_url(REPO_URL)

if os.path.exists(PROJECT_DIR):
    print(f"ℹ️ Removing existing folder: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

print(f"Cloning into {PROJECT_DIR} ...")
result = subprocess.run(
    ["git", "clone", clone_url, PROJECT_DIR],
    text=True,
    capture_output=True,
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        "git clone failed. If the repo is private, add GITHUB_TOKEN as a Colab Secret and rerun Cell 1."
    )

print(f"✅ Cloned to {PROJECT_DIR}")
os.chdir(PROJECT_DIR)

!pip install -q mlflow boto3 pyarrow pandas numpy scikit-learn xgboost lightgbm
print("\n✅ Dependencies installed")

Cloning into /content/053_memory_yield_predictor ...
✅ Cloned to /content/053_memory_yield_predictor
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: Set AWS Credentials for S3 Upload
# ═══════════════════════════════════════════════════════════════
# Option A: Use Colab Secrets (recommended — go to 🔑 icon in left panel)
# Option B: Paste directly (NOT recommended for shared notebooks)

try:
    from google.colab import userdata
    os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
    os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
    print("✅ AWS credentials loaded from Colab Secrets")
except Exception:
    print("⚠️  Colab Secrets not set. Set them manually:")
    print("    os.environ['AWS_ACCESS_KEY_ID'] = 'your-key'")
    print("    os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret'")

# Always set these
os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'
os.environ['S3_BUCKET'] = 'p053-mlflow-artifacts'
os.environ['COMPUTE_BACKEND'] = 'colab'
os.environ['MODEL_PARAMS'] = '317000'
os.environ['SIMULATION_SCALE'] = 'phase2'

# Verify S3 access
try:
    import boto3
    s3 = boto3.client('s3', region_name='us-west-2')
    resp = s3.list_objects_v2(Bucket='p053-mlflow-artifacts', MaxKeys=3)
    n = resp.get('KeyCount', 0)
    print(f"✅ S3 connected: {n} objects found in p053-mlflow-artifacts")
except Exception as e:
    print(f"❌ S3 connection failed: {e}")
    print("   Check your AWS credentials above.")

⚠️  Colab Secrets not set. Set them manually:
    os.environ['AWS_ACCESS_KEY_ID'] = 'your-key'
    os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret'
❌ S3 connection failed: Unable to locate credentials
   Check your AWS credentials above.


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: Verify GPU + Hardware Detection
# ═══════════════════════════════════════════════════════════════
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    cc = torch.cuda.get_device_capability(0)
    bf16 = cc[0] >= 8
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    print(f"Compute Capability: {cc[0]}.{cc[1]}")
    print(f"bfloat16 support: {bf16}")
    print(f"\n✅ GPU ready for training")

    if 'T4' in gpu_name:
        print("   → Using float16 + GradScaler (T4 compute cap 7.5)")
        print("   → Estimated: ~1.36 CU/hr")
    elif 'A100' in gpu_name:
        print("   → Using bfloat16, NO GradScaler (A100 compute cap 8.0)")
        print("   → Estimated: ~6.79 CU/hr")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

# Test compute backend
from src.compute_backend import get_training_backend
backend = get_training_backend()
print(f"\nCompute Backend: {backend.name.value}")
print(f"  GPU: {backend.gpu_name}")
print(f"  MLflow: {backend.mlflow_uri}")
print(f"  Batch size: {backend.recommended_batch_size}")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB
Compute Capability: 7.5
bfloat16 support: False

✅ GPU ready for training
   → Using float16 + GradScaler (T4 compute cap 7.5)
   → Estimated: ~1.36 CU/hr

Compute Backend: colab
  GPU: Tesla T4
  MLflow: sqlite:///mlflow.db
  Batch size: 4096


In [4]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Mount Drive + Copy preprocessed data
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
from pathlib import Path
import shutil

DRIVE_INPUT_DIR = '/content/drive/MyDrive/P053_data'
PROJECT_DIR = '/content/053_memory_yield_predictor'
NPZ_DEST = f'{PROJECT_DIR}/data/preprocessed_full.npz'

# Mount once for both reading input data and writing outputs
drive.mount('/content/drive')
Path(NPZ_DEST).parent.mkdir(parents=True, exist_ok=True)

candidates = [
    f'{DRIVE_INPUT_DIR}/preprocessed_full.npz',
    '/content/drive/MyDrive/preprocessed_full.npz',
]

for candidate in candidates:
    if Path(candidate).exists():
        shutil.copy2(candidate, NPZ_DEST)
        print(f'✅ Copied preprocessed_full.npz from {candidate}')
        print(f'   Size: {Path(NPZ_DEST).stat().st_size / 1e6:.1f} MB')
        break
else:
    print('⚠️ preprocessed_full.npz not found on Drive.')
    print('   Simulation still runs, but retraining becomes metadata-only.')
    print(f'   Checked: {candidates}')

Mounted at /content/drive
✅ Copied preprocessed_full.npz from /content/drive/MyDrive/P053_data/preprocessed_full.npz
   Size: 2232.0 MB


In [8]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: Run all 3 modes + backup after each
# ═══════════════════════════════════════════════════════════════
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

PROJECT_DIR = '/content/053_memory_yield_predictor'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/P053_results'
Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

RUNS = [
    {'name': 'fast', 'rows': 100_000, 'epochs': 0},
    {'name': 'medium', 'rows': 1_000_000, 'epochs': 10},
    {'name': 'full', 'rows': 5_000_000, 'epochs': 10},
]

results_summary = []

for run_cfg in RUNS:
    run_name = run_cfg['name']
    rows = run_cfg['rows']
    epochs = run_cfg['epochs']

    print(f"\n{'=' * 70}")
    print(f"STARTING {run_name.upper()} | rows/day={rows:,} | sim_retrain_epochs={epochs}")
    print(f"{'=' * 70}")

    checkpoint_path = Path(PROJECT_DIR) / 'data' / 'simulation_checkpoint.json'
    checkpoint_path.unlink(missing_ok=True)

    cmd = [
        sys.executable,
        '-m',
        'src.run_simulation',
        '--rows',
        str(rows),
        '--backend',
        'colab',
        '--checkpoint',
    ]

    # The --sim-retrain-epochs argument is not recognized by src.run_simulation.
    # It's removed here to fix the 'unrecognized arguments' error.
    # If retraining epochs are needed, they should be configured differently or
    # confirmed to be supported by the simulation script.

    print(f"DEBUG: Command for subprocess: {cmd}") # Added for debugging

    t0 = time.time()
    result = subprocess.run(cmd, cwd=PROJECT_DIR, capture_output=True, text=True)
    elapsed_min = (time.time() - t0) / 60

    if result.returncode != 0:
        print(f"--- {run_name.upper()} ERROR DETAILS ---")
        if result.stdout:
            print("Standard Output:")
            print(result.stdout)
        if result.stderr:
            print("Standard Error:")
            print(result.stderr)
        print(f"----------------------------------")
        raise RuntimeError(f'{run_name} run failed with exit code {result.returncode}')

    backup_dir = Path(DRIVE_OUTPUT_DIR) / run_name
    backup_dir.mkdir(parents=True, exist_ok=True)

    files_to_backup = [
        'data/simulation_timeline.json',
        'data/simulation_checkpoint.json',
        'mlflow.db',
    ]
    for rel_path in files_to_backup:
        src = Path(PROJECT_DIR) / rel_path
        if src.exists():
            dest_name = src.name
            if src.name == 'mlflow.db':
                dest_name = f'mlflow_{run_name}.db'
            shutil.copy2(src, backup_dir / dest_name)

    for benchmark in (Path(PROJECT_DIR) / 'data').glob('benchmark_*.json'):
        shutil.copy2(benchmark, backup_dir / benchmark.name)

    drift_src = Path(PROJECT_DIR) / 'data' / 'drift_reports'
    if drift_src.exists():
        drift_dest = backup_dir / 'drift_reports'
        if drift_dest.exists():
            shutil.rmtree(drift_dest)
        shutil.copytree(drift_src, drift_dest)

    mlruns_src = Path(PROJECT_DIR) / 'mlruns'
    if mlruns_src.exists():
        mlruns_dest = backup_dir / 'mlruns'
        if mlruns_dest.exists():
            shutil.rmtree(mlruns_dest)
        shutil.copytree(mlruns_src, mlruns_dest)

    timeline_path = Path(PROJECT_DIR) / 'data' / 'simulation_timeline.json'
    retrains = 0
    total_days = 0
    if timeline_path.exists():
        with open(timeline_path) as f:
            timeline = json.load(f)
        retrains = len(timeline.get('retrain_events', []))
        total_days = timeline.get('total_days', 0)

    results_summary.append({
        'name': run_name,
        'rows': rows,
        'epochs': epochs,
        'elapsed_min': elapsed_min,
        'retrains': retrains,
        'total_days': total_days,
        'backup_dir': str(backup_dir),
    })

    print(f"✅ {run_name.upper()} complete")
    print(f"   Days: {total_days} | Retrains: {retrains} | Time: {elapsed_min:.1f} min")
    print(f"   Saved to: {backup_dir}")

summary_path = Path(PROJECT_DIR) / 'data' / 'colab_run_summary.json'
with open(summary_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"\nSaved run summary to {summary_path}")


STARTING FAST | rows/day=100,000 | sim_retrain_epochs=0
DEBUG: Command for subprocess: ['/usr/bin/python3', '-m', 'src.run_simulation', '--rows', '100000', '--backend', 'colab', '--checkpoint']
✅ FAST complete
   Days: 40 | Retrains: 1 | Time: 8.5 min
   Saved to: /content/drive/MyDrive/P053_results/fast

STARTING MEDIUM | rows/day=1,000,000 | sim_retrain_epochs=10
DEBUG: Command for subprocess: ['/usr/bin/python3', '-m', 'src.run_simulation', '--rows', '1000000', '--backend', 'colab', '--checkpoint']
✅ MEDIUM complete
   Days: 40 | Retrains: 1 | Time: 11.8 min
   Saved to: /content/drive/MyDrive/P053_results/medium

STARTING FULL | rows/day=5,000,000 | sim_retrain_epochs=10
DEBUG: Command for subprocess: ['/usr/bin/python3', '-m', 'src.run_simulation', '--rows', '5000000', '--backend', 'colab', '--checkpoint']
✅ FULL complete
   Days: 40 | Retrains: 1 | Time: 29.9 min
   Saved to: /content/drive/MyDrive/P053_results/full

Saved run summary to /content/053_memory_yield_predictor/data/

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: Final summary after all 3 runs
# ═══════════════════════════════════════════════════════════════
import json
from pathlib import Path

summary_path = Path('/content/053_memory_yield_predictor/data/colab_run_summary.json')

if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)

    print('=' * 70)
    print('ALL RUNS COMPLETE')
    print('=' * 70)
    total_minutes = 0.0
    for item in summary:
        total_minutes += item['elapsed_min']
        print(
            f"{item['name']:<8} days={item['total_days']:<2} "
            f"retrains={item['retrains']:<2} "
            f"time={item['elapsed_min']:.1f} min"
        )
        print(f"         backup: {item['backup_dir']}")
    print('-' * 70)
    print(f'TOTAL TIME: {total_minutes:.1f} min')
    print('Most important output to copy back later: MyDrive/P053_results/full/')
else:
    print('No summary file found. Run Cell 5 first.')

---

## What To Do

Run Cells 1 to 5 once.
Then run Cell 6 to confirm all three modes finished.

Outputs are created automatically:
- `MyDrive/P053_results/fast/`
- `MyDrive/P053_results/medium/`
- `MyDrive/P053_results/full/`

Copy back only the `full/` folder to the workspace when done.
Then run:

```bash
python -m src.post_simulation_update
```

If Colab disconnects, run the notebook again. The current mode resumes from its checkpoint.

---
*P053 — AIML Engineering Lab*